### Silver Helpers
This notebook has a reusable function called `write_to_silver()` that all silver notebooks use to save their cleaned data.

---

#### What does `write_to_silver()` do?

It handles **two situations**:

**1. First time run (table doesn't exist yet)**
- Simply creates the silver table and writes all the data into it
- Adds `created_timestamp` and `updated_timestamp` columns automatically

**2. Table already exists (incremental load)**
- Uses Delta Lake **MERGE** to compare new data with existing data
- If a row already exists (matched by the merge condition) AND the new batch is newer → **updates** that row
- If a row is brand new (not matched) → **inserts** it
- This way we never lose old data and always have the latest version

---

#### Parameters it takes:

| Parameter | What it means |
| --- | --- |
| `input_df` | The cleaned DataFrame you want to save |
| `target_table` | Full table name like `formula1_incr.silver.circuits` |
| `merge_condition` | How to match rows, e.g. `t.circuit_id = s.circuit_id` |
| `columns_to_update` | List of columns to update when a match is found |

---

#### How the MERGE works (step by step):
1. Your new data gets alias `s` (source), existing table gets alias `t` (target)
2. Spark compares rows using your `merge_condition`
3. **If matched** AND `s.batch_id >= t.batch_id` → updates only the columns you listed + `updated_timestamp`
4. **If not matched** → inserts the entire new row
5. `created_timestamp` stays the same (never changes after first insert)
6. `updated_timestamp` changes every time a row is updated

---

#### Why we use this helper:
- Avoids repeating the same merge code in every notebook
- Makes sure all silver tables follow the same pattern
- Handles both first-run and incremental-run automatically

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
def write_to_silver(
    input_df,
    target_table,
    merge_condition,
    columns_to_update
):
    
    final_df = (
        input_df
        .withColumn('created_timestamp', current_timestamp())
        .withColumn('updated_timestamp', current_timestamp())
    )
    if not spark.catalog.tableExists(target_table):
        (
            final_df
            .write
            .mode("overwrite")
            .format("delta")
            .saveAsTable(target_table)
        )
    else:
        delta_table = DeltaTable.forName(spark, target_table)
        update_map = {column: f's.{column}' for column in columns_to_update}
        update_map['updated_timestamp'] = 's.updated_timestamp'
        (
            delta_table.alias('t')
            .merge(
                final_df.alias('s'),
                merge_condition
            )
            .whenMatchedUpdate(
                condition = 's.batch_id >= t.batch_id',
                set = update_map
            )
            .whenNotMatchedInsertAll()
            .execute()
        )